# chain-rule-elementwise — worked example 3: Backward of out = x**2 with a per-row grad_out scale

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `chain-rule-elementwise`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The squaring op `out = x**2` is elementwise with local derivative `f'(x) = 2*x`. So `grad_in = grad_out * 2 * x`, shape-preserving and matmul-free. This worked example also shows that an upstream `grad_out` of any compatible shape just multiplies position-by-position — the diagonal-Jacobian structure never changes.

## Worked solution

**Step 1 — forward and derivative.** `out = x**2` elementwise, and `d/dx x**2 = 2*x`. The derivative depends on `x`, not `out` (you can't sign-recover `x` from `x**2`).

**Step 2 — build a structured grad_out.** Suppose the loss is a row-weighted sum, so the upstream gradient is constant within each row. We construct `grad_out` by `repeat`-ing a per-row weight vector across columns with einops `repeat`. This is still the same shape as `x`, so nothing about the chain rule changes.

**Step 3 — apply the elementwise chain rule.** `grad_in = grad_out * (2 * x)`. Each position multiplies its own upstream gradient by its own local slope `2*x` — no cross-position mixing.

**Step 4 — verify against autograd.** We replay `x**2`, backprop the same `grad_out`, and compare. The diagonal-Jacobian product reproduces autograd exactly up to roundoff, confirming that even a non-uniform upstream gradient stays a pure elementwise multiply.

In [ ]:
def square_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    # d/dx x**2 = 2*x. Depends on x (sign is lost in out).
    return grad_out * 2 * x


t.manual_seed(0)
x = t.randn(4, 3, dtype=t.float64)
row_w = t.randn(4, dtype=t.float64)
# Same upstream weight across every column of a row -> shape (4, 3).
grad_out = repeat(row_w, "r -> r c", c=3)
out = x ** 2
grad_in = square_back(grad_out, out, x)

xg = x.clone().requires_grad_(True)
(xg ** 2).backward(grad_out)
print("max abs diff vs autograd:", (grad_in - xg.grad).abs().max().item())
print("grad_in shape:", tuple(grad_in.shape))